In [0]:
catalog = "workspace"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

In [0]:
from pyspark.sql import functions as F, Row
from datetime import datetime
from pyspark.sql.window import Window
dq_results = []

def dq_check(table_name: str, check_name: str, df, condition):
    """Executa uma checagem de qualidade: conta quantas linhas violam a condição esperada."""
    total = df.count()
    failed = df.filter(~condition).count()
    passed = failed == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=failed, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {failed}/{total} linhas falharam")

def dq_check_unique(table_name: str, check_name: str, df, key_cols: list):
    """Checagem de qualidade específica para unicidade de chave."""
    total = df.count()
    dupes = df.groupBy(*key_cols).count().filter("count > 1").count()
    passed = dupes == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=dupes, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {dupes} chaves duplicadas de {total} linhas")


In [0]:
df_bronze_info = spark.table(f"{catalog}.bronze.tb_movies_info")

display(spark.table(f"{catalog}.bronze.tb_movies_info"))

Tratamento de Informações do Filme: Deduplicação de registros priorizando a data de ingestão mais recente. Tratamento de múltiplos formatos de data inconsistentes na origem e padronização da coluna de status (tradução do inglês para o português e remoção de caracteres inválidos).

In [0]:

df_silver_info = (
    df_bronze_info
    .withColumn("rank", F.row_number().over(Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc())))
    .filter(F.col("rank") == 1)
    .drop("rank")
    .withColumn("status",F.upper(F.trim(F.regexp_replace(F.col("status"),"-", ""))))
    .withColumn("release_date", F.coalesce(
        F.expr("try_to_date(release_date, 'yyyy-MM-dd')"), 
        F.expr("try_to_date(release_date, 'dd/MM/yyyy')"), 
        F.expr("try_to_date(release_date, 'MM/dd/yyyy')"),
        F.expr("try_to_date(release_date, 'MM-dd-yyyy')"),
        F.expr("try_to_date(release_date, 'dd-MM-yyyy')"),
        F.expr("try_to_date(release_date, 'yyyy/MM/dd')")
    ))
    .withColumn("status",
                F.when(F.col("status") == "RELEASED", "Lançado")
                .when(F.col("status") == "POSTPRODUCTION", "Pós-Produção")
                .when(F.col("status") == "IN PRODUCTION", "Em produção")
                .when(F.col("status") == "PLANNED", "Planejado")
                .when(F.col("status") == "RUMORED", "Rumores")
                .when(F.col("status") == "CANCELED", "Cancelado")
                .otherwise("Não Informado"))
    .withColumn("ano_lancamento",F.year(F.col("release_date")))
    .select(
        F.expr("try_cast(id as int)").alias("id_filme"),
        F.col("title").alias("titulo"),
        F.col("original_title").alias("titulo_original"),
        F.col("release_date").alias("data_lancamento"),
        F.expr("try_cast(runtime as int)").alias("duracao_minutos"), 
        F.col("original_language").alias("idioma_original"),
        F.col("status").alias("status_filme"),
        F.col("overview").alias("sinopse"),
        F.col("tagline").alias("frase_divulgacao"),
        F.col("ano_lancamento")
    )
)

dq_check("silver.info", "id_filme não nulo", df_silver_info, F.col("id_filme").isNotNull())
dq_check_unique("silver.info", "id_filme único", df_silver_info, ["id_filme"])

df_silver_info.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.info")

display(df_bronze_info.limit(10))
display(df_silver_info.limit(10))





In [0]:
df_bronze_financials = spark.table(f"{catalog}.bronze.tb_movies_financials")

display(spark.table(f"{catalog}.bronze.tb_movies_financials"))

df_bronze_cotacao = spark.table(f"{catalog}.bronze.tb_cotacao_dolar")

display(spark.table(f"{catalog}.bronze.tb_cotacao_dolar").limit(10))

taxa_cotacao_dinamica = (
    df_bronze_cotacao
    .orderBy(F.col("ingestion_datetime").desc())
    .select("cotacaoCompra")
    .first()["cotacaoCompra"]
)

Normalização Financeira: Limpeza de sufixos monetários (USD, $) e conversão de multiplicadores de texto (extração de 'M' para milhões e 'K' para milhares) em valores decimais reais. Inclui a conversão cambial dinâmica para BRL (Reais) e o cálculo de métricas derivadas (lucro e margem percentual), anulando orçamentos ou receitas negativas/nulas oriundas da extração.

In [0]:
df_silver_financials = (
    df_bronze_financials
    .withColumn("budget", 
        F.when(F.trim(F.col("budget")).isin("", "N/A"), F.lit(None))
        .otherwise(F.trim(F.regexp_replace(F.col("budget"), r"[$,\s]|(?i)USD", "")))
    )
    .withColumn("revenue", 
        F.when(F.trim(F.col("revenue")).isin("Unknown", "Não Informado", "null", "",), F.lit(None))
        .otherwise(F.trim(F.regexp_replace(F.col("revenue"), r"[$,\s]|(?i)USD", "")))
    )
    .withColumn("orcamento_usd", 
        F.when(F.lower(F.col("budget")).endswith("m"), 
               F.expr("try_cast(regexp_replace(budget, '(?i)M', '') as decimal(24,2))") * 1000000)
        .when(F.lower(F.col("budget")).endswith("k"), 
               F.expr("try_cast(regexp_replace(budget, '(?i)K', '') as decimal(24,2))") * 1000)
        .otherwise(F.expr("try_cast(budget as decimal(24,2))"))
    )
    .withColumn("receita_usd", F.expr("try_cast(revenue as decimal(24,2))"))

    .withColumn("orcamento_usd", F.when(F.col("orcamento_usd") <= 0, F.lit(None)).otherwise(F.col("orcamento_usd")))
    .withColumn("receita_usd", F.when(F.col("receita_usd") <= 0, F.lit(None)).otherwise(F.col("receita_usd")))

    .withColumn("orcamento_brl", (F.col("orcamento_usd") * F.lit(taxa_cotacao_dinamica)).cast("decimal(24,2)"))

    .withColumn("receita_brl", (F.col("receita_usd") * F.lit(taxa_cotacao_dinamica)).cast("decimal(24,2)"))

    .withColumn("lucro_usd", F.col("receita_usd") - F.col("orcamento_usd"))

    .withColumn("lucro_brl", F.col("receita_brl") - F.col("orcamento_brl"))
    .withColumn("margem_lucro_percentual", 
        F.when(F.col("orcamento_usd") > 0, 
               ((F.col("lucro_usd") / F.col("orcamento_usd")) * 100).cast("decimal(18,2)")
        ).otherwise(F.lit(None))
    )
    
    .select(
        F.expr("try_cast(id as int)").alias("id_filme"),
        "orcamento_usd",
        "receita_usd",
        "orcamento_brl",
        "receita_brl",
        "lucro_usd",
        "lucro_brl",
        "margem_lucro_percentual"
    )
)
dq_check("silver.financials", "id_filme", df_silver_financials, F.col("id_filme").isNotNull())
dq_check("silver.financials", "orcamento_usd não negativo", df_silver_financials,
         F.col("orcamento_usd") >= 0)
display(df_silver_financials.limit(15))
df_silver_financials.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.financials")

In [0]:
df_bronze_metrics = spark.table(f"{catalog}.bronze.tb_movies_metrics")
display(spark.table(f"{catalog}.bronze.tb_movies_metrics"))

Qualidade de Métricas: Adequação da tipagem de dados numéricos substituindo separadores mal formatados. Aplicação de regras estritas de qualidade de dados: validação para garantir que as notas variem apenas no intervalo de 0 a 10 e remoção de contagens de votos com valores negativos.

In [0]:
df_silver_metrics = (
    df_bronze_metrics
    .withColumn("popularity", F.regexp_replace(F.col("popularity"),",","."))
    .withColumn("populartity",F.expr("try_cast(popularity as decimal(18,3))"))
    .withColumn("vote_average",F.expr("try_cast(vote_average as decimal(4,2))"))
    .withColumn("vote_count",F.expr("try_cast(vote_count as int)"))
    .withColumn("averageRating",F.expr("try_cast(averageRating as decimal(4,2))"))
    .withColumn("numVotes",F.expr("try_cast(numVotes as int)"))
    .withColumn("nota_media_tmdb",F.when((F.col("vote_average")>=0) & (F.col("vote_average")<=10),F.col("vote_average")).otherwise(F.lit(None)))
    .withColumn("nota_media_imdb",F.when((F.col("averageRating") >=0)&(F.col("averageRating") <=10),F.col("averageRating")).otherwise(F.lit(None)))

    .withColumn("qtd_votos_tmdb",F.when(F.col("vote_count") >=0,F.col("vote_count")).otherwise(F.lit(None)))
    .withColumn("qtd_votos_imdb",F.when(F.col("numVotes") >=0,F.col("numVotes")).otherwise(F.lit(None)))
    .withColumn("popularidade",F.when(F.col("populartity") >=0,F.col("populartity")).otherwise(F.lit(None)))
   
    .select(
        F.expr("try_cast(id as int)").alias("id_filme"),
        "popularidade",
        "nota_media_tmdb",
        "qtd_votos_tmdb",
        "nota_media_imdb",
        "qtd_votos_imdb"
    )
)
dq_check("silver.metrics", "id_filme", df_silver_metrics, F.col("id_filme").isNotNull())
dq_check("silver.metrics", "id_filme > 0", df_silver_metrics, F.col("id_filme") > 0)
display(df_silver_metrics.limit(15))
df_silver_metrics.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.metrics")
        

In [0]:
df_bronze_reviews = spark.table(f"{catalog}.bronze.tb_movies_reviews")
display(spark.table(f"{catalog}.bronze.tb_movies_reviews"))


Tratamento de Comentários: Deduplicação de comentários idênticos feitos pelo mesmo usuário em um único filme. Validação da nota atribuída (limitada entre 0 e 10) e padronização de campos de texto vazios/nulos com o padrão "Sem comentário".

In [0]:
df_silver_reviews = (
    df_bronze_reviews
    .dropDuplicates(["id","nome","comentario","nota"])
    .withColumn("nota_usuario",
         F.when((F.col("nota") >= 0)&(F.col("nota")<=10),F.col("nota"))
         .otherwise(F.lit(None)))
    .withColumn("comentario_usuario", 
        F.when(F.col("comentario").isNull() | (F.trim(F.col("comentario")) == ""), F.lit("Sem comentário"))
        .otherwise(F.trim(F.col("comentario")))
    )
    .select(
        F.col("id").alias("id_filme"),
        F.col("nome").alias("nome_usuario"),
        "nota_usuario",
        "comentario_usuario"
    )
    )
dq_check("silver.reviews", "id_filme", df_silver_reviews, F.col("id_filme").isNotNull())
dq_check("silver.reviews", "nota_usuario não negativo", df_silver_reviews,
         F.col("nota_usuario") >= 0)
display(df_silver_reviews.limit(15))
df_silver_reviews.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.reviews")

In [0]:
df_bronze_generos = spark.table(f"{catalog}.bronze.tb_credits_and_tags")
display(spark.table(f"{catalog}.bronze.tb_credits_and_tags"))


Separação de Gêneros: Limpeza da formatação de listas em string (remoção de colchetes) e pivotamento de múltiplos gêneros em uma única coluna usando a função explode. Aplicação de um filtro rigoroso de domínio cruzando os dados contra uma lista restrita de gêneros válidos de streaming, impedindo a entrada de categorias errôneas no modelo Gold.

In [0]:
streaming_genres = [
    "Action", "Adventure", "Animation", "Comedy", "Crime", "Documentary", 
    "Drama", "Family", "Fantasy", "History", "Horror", "Music", "Mystery", 
    "Romance", "Science Fiction", "TV Movie", "Thriller", "War", "Western"
]
df_silver_generos = (
    df_bronze_generos
    .withColumn("genero", F.regexp_replace(F.col("genres"), r'[\[\]"]', ""))
    .withColumn("genero", F.explode(F.split(F.col("genero"), "[,|;]")))
    .withColumn("genero", F.trim(F.col("genero")))
        .filter(F.col("genero").isin(streaming_genres))
    .select(
        F.col("id").alias("id_filme").cast("int"),
        F.col("genero")
    )
    .dropDuplicates()
)
dq_check("silver.generos", "id_filme não nulo", df_silver_generos, F.col("id_filme").isNotNull())
dq_check_unique("silver.generos", "id_filme e genero únicos", df_silver_generos, ["id_filme", "genero"])
display(df_silver_generos.limit(15))
df_silver_generos.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.silver.generos")

Normalização de Entidades (Elenco e Produção): Pivotamento das matrizes de atores, diretores, roteiristas e produtoras para um formato longo. Aplicação de regras para exclusão de placeholders (ex: 'N/a'), remoção de artefatos de extração (arquivos .jpg) e restrição do limite de caracteres com base no cargo de cada entidade.

In [0]:
df_silver_pessoas_empresas = (
    df_bronze_generos
    .select(
        F.col("id").alias("id_filme"),
        F.expr("""
            stack(4, 
                'Ator', cast, 
                'Diretor', directors, 
                'Roteirista', writers, 
                'Produtora', production_companies
            ) as (tipo_entidade, nomes)
        """)
    )
    .withColumn("nome", F.explode(F.split(F.regexp_replace(F.col("nomes"), r"[;|]", ","), ",")))
    .withColumn("nome", F.trim(F.col("nome")))
    .withColumn("nome", F.regexp_replace(F.col("nome"), r"^['\".,\-()*]+|['\".,\-()*]+$", ""))
    .withColumn("nome", F.initcap(F.trim(F.col("nome"))))
    .filter(
        F.col("nome").isNotNull() & 
        (F.col("nome") != "") & 
        (F.length(F.col("nome")) > 2) & 
        (~F.col("nome").startswith("/")) &
        (~F.col("nome").rlike(r'(?i)\.(jpg|jpeg|png|gif)')) &
        (~F.lower(F.col("nome")).isin("n/a", "nenhum", "none", "null", "na")) &
        (
            (
                (F.col("tipo_entidade") == "Produtora") &
                (F.length(F.col("nome")) <= 80)
            ) 
            |
            (
                (F.col("tipo_entidade") != "Produtora") &
                (F.length(F.col("nome")) <= 35) &
                (~F.col("nome").rlike(r'\d')) &
                (~F.col("nome").startswith("@")) &
                (~F.col("nome").rlike(r'(?i)(immigrant|driver|senator|novelist|group|time ago|century|films|pictures|throwback|broken life|president|artist|styled|land of war|story|journey)')) &
                (~F.col("nome").rlike(r'^(A|An)\s+'))
            )
        )
    )
    .select("id_filme","nome", "tipo_entidade")
    .dropDuplicates()
)
dq_check("silver.pessoas_empresas", "nome não nulo", df_silver_pessoas_empresas, F.col("nome").isNotNull())

display(df_silver_pessoas_empresas.limit(20))
df_silver_pessoas_empresas.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.pessoas_empresas")

In [0]:
display(spark.table(f"{catalog}.bronze.tb_cotacao_dolar"))

Interpolação de Séries Temporais Cambiais: Geração de um calendário contínuo para tratar os buracos nos dados referentes a finais de semana e feriados sem cotação oficial de mercado. Utilização de uma Window Function (Forward Fill) para projetar o último valor de fechamento válido para os dias ausentes, garantindo uma conversão exata na data de lançamento de qualquer filme

In [0]:
df_bronze = spark.table(f"{catalog}.bronze.tb_cotacao_dolar")

df_bronze_tratada = (
    df_bronze
    .withColumn("data", F.to_date(F.col("dataHoraCotacao")))
    .select(
        F.col("data"),
        F.col("cotacaoCompra")
    )
    .dropDuplicates() 
)


datas_limites = df_bronze_tratada.select(
    F.min("data").alias("min_data"), 
    F.max("data").alias("max_data")
).collect()[0]

min_data = datas_limites["min_data"]
max_data = datas_limites["max_data"]


df_calendario_continuo = spark.sql(f"""
    SELECT explode(sequence(to_date('{min_data}'), to_date('{max_data}'), interval 1 day)) as data_continua
""")


df_com_buracos = (
    df_calendario_continuo
    .join(df_bronze_tratada, df_calendario_continuo["data_continua"] == df_bronze_tratada["data"], "left")
    .select(
        F.col("data_continua").alias("data"),
        F.col("cotacaoCompra")
    )
)


janela_ffill = Window.partitionBy(F.lit(1)).orderBy("data").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_silver_cotacao_dolar = (
    df_com_buracos
    .withColumn("cotacao_preenchida", F.last("cotacaoCompra", ignorenulls=True).over(janela_ffill))
    .select(
        "data",
        F.col("cotacao_preenchida").alias("cotacao_compra")
    )
)

dq_check("silver.cotacao_dolar", "data não nulo", df_silver_cotacao_dolar, F.col("data").isNotNull())
dq_check("silver.cotacao_dolar", "preco > 0", df_silver_cotacao_dolar, F.col("cotacao_compra") > 0)
display(df_silver_cotacao_dolar.limit(20))
df_silver_cotacao_dolar.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.cotacao_dolar")